# VibeShift Model Test Suite

Comprehensive test suite for testing the DiT (Diffusion Transformer) and FlowMatching models with DAC audio codec integration.

## Section 1: Install Dependencies and Import Libraries

Install required packages and import all necessary libraries for the test suite.

In [ ]:
# Install dependencies
!pip install descript-audio-codec torch torchaudio omegaconf

import torch
import torch.nn as nn
import dac
from pathlib import Path
import traceback
import time

print("✓ All dependencies installed and imported successfully")

## Section 2: Test 1 - Model Instantiation

Create instances of the DiT and FlowMatching models with explicit parameters and verify successful instantiation.

In [ ]:
print("=" * 50)
print("TEST 1: Model Instantiation")
print("=" * 50)

try:
    from models.dit import DiT
    from models.flow import FlowMatching
    
    # Create model with explicit parameters (no config file needed)
    dit = DiT(
        input_dim=64,  # Use DAC's actual latent dim (check with DAC model)
        embed_dim=512,
        num_blocks=4,  # Small for testing
        num_heads=8,
        num_genres=3,
        hidden_dim=2048,
        dropout=0.1
    )
    
    flow_model = FlowMatching(dit)
    
    print("✓ Models instantiated successfully")
    print(f"  - DiT parameters: {sum(p.numel() for p in dit.parameters()):,}")
    print(f"  - Flow model parameters: {sum(p.numel() for p in flow_model.parameters()):,}")
    
except Exception as e:
    print(f"✗ Model instantiation failed: {e}")
    traceback.print_exc()

## Section 3: Test 2 - DAC Audio Codec Integration

Download and load the DAC model, test encoding/decoding with dummy audio, and identify the actual latent dimension.

In [ ]:
print("\n" + "=" * 50)
print("TEST 2: DAC Audio Codec Integration")
print("=" * 50)

try:
    # Download and load DAC model
    model_path = dac.utils.download(model_type="44khz")
    dac_model = dac.DAC.load(model_path)
    dac_model.eval()
    
    # Test with dummy audio (1 second at 44.1kHz)
    dummy_audio = torch.randn(2, 1, 44100)
    print(f"Input audio shape: {dummy_audio.shape}")
    
    with torch.no_grad():
        z, codes, latents, _, _ = dac_model.encode(dummy_audio)
    
    print(f"✓ DAC encoding successful")
    print(f"  - Latent shape (DAC format): {z.shape}")
    print(f"  - Expected format: (B, latent_dim, T)")
    
    # Transpose for your model
    z_transposed = z.transpose(1, 2)
    print(f"  - Latent shape (VibeShift format): {z_transposed.shape}")
    print(f"  - Expected format: (B, T, latent_dim)")
    
    # Update your DiT model with correct latent_dim
    actual_latent_dim = z.shape[1]
    print(f"\n⚠️  IMPORTANT: Update input_dim={actual_latent_dim} in your DiT model")
    
    # Test decoding
    z_back = z_transposed.transpose(1, 2)
    with torch.no_grad():
        reconstructed = dac_model.decode(z_back)
    print(f"✓ DAC decoding successful: {reconstructed.shape}")
    
except Exception as e:
    print(f"✗ DAC test failed: {e}")
    traceback.print_exc()

## Section 4: Test 3 - Model Forward Pass

Create dummy inputs with correct shapes and verify forward pass output matches input shape.

In [ ]:
print("\n" + "=" * 50)
print("TEST 3: Model Forward Pass")
print("=" * 50)

try:
    batch_size = 2
    seq_len = 100  # Time steps after DAC encoding
    latent_dim = actual_latent_dim  # From DAC test
    
    # Create dummy inputs
    x = torch.randn(batch_size, seq_len, latent_dim)
    t = torch.rand(batch_size)
    genre_ids = torch.tensor([0, 1])  # Classical and Rock
    
    print(f"Input shapes:")
    print(f"  - x: {x.shape}")
    print(f"  - t: {t.shape}")
    print(f"  - genre_ids: {genre_ids.shape}")
    
    # Forward pass through DiT
    with torch.no_grad():
        output = dit(x, t, genre_ids)
    
    print(f"\n✓ DiT forward pass successful")
    print(f"  - Output shape: {output.shape}")
    print(f"  - Expected: {x.shape}")
    
    assert output.shape == x.shape, "Output shape mismatch!"
    print("✓ Shape assertion passed")
    
except Exception as e:
    print(f"✗ Forward pass failed: {e}")
    traceback.print_exc()

## Section 5: Test 4 - Flow Matching Loss Computation

Compute flow matching loss, verify it's scalar, and check for NaN/Inf values.

In [ ]:
print("\n" + "=" * 50)
print("TEST 4: Flow Matching Loss")
print("=" * 50)

try:
    # Create source and target embeddings
    x0 = torch.randn(batch_size, seq_len, latent_dim)
    x1 = torch.randn(batch_size, seq_len, latent_dim)
    genre_ids = torch.tensor([1, 1])  # Both rock
    
    # Compute loss
    loss = flow_model.compute_loss(x0, x1, genre_ids)
    
    print(f"✓ Loss computation successful")
    print(f"  - Loss value: {loss.item():.6f}")
    print(f"  - Loss shape: {loss.shape}")
    
    assert loss.numel() == 1, "Loss should be scalar!"
    assert not torch.isnan(loss), "Loss is NaN!"
    assert not torch.isinf(loss), "Loss is Inf!"
    print("✓ Loss validation passed")
    
except Exception as e:
    print(f"✗ Loss computation failed: {e}")
    traceback.print_exc()

## Section 6: Test 5 - Flow Matching Sampling

Test both Euler and Heun sampling methods for genre transformation.

In [ ]:
print("\n" + "=" * 50)
print("TEST 5: Flow Matching Sampling")
print("=" * 50)

try:
    x0 = torch.randn(batch_size, seq_len, latent_dim)
    target_genre = 1  # Rock
    
    # Test Euler sampling
    print("Testing Euler method...")
    with torch.no_grad():
        x_transformed_euler = flow_model.sample_euler(x0, target_genre, num_steps=10)
    
    print(f"✓ Euler sampling successful")
    print(f"  - Output shape: {x_transformed_euler.shape}")
    
    # Test Heun sampling
    print("\nTesting Heun method...")
    with torch.no_grad():
        x_transformed_heun = flow_model.sample_heun(x0, target_genre, num_steps=10)
    
    print(f"✓ Heun sampling successful")
    print(f"  - Output shape: {x_transformed_heun.shape}")
    
    # Check outputs are different from input
    assert not torch.allclose(x0, x_transformed_euler), "Output should differ from input!"
    print("✓ Transformation validation passed")
    
except Exception as e:
    print(f"✗ Sampling failed: {e}")
    traceback.print_exc()

## Section 7: Test 6 - Training Step Simulation

Simulate a complete training step with forward pass, loss computation, backward pass, and optimization.

In [ ]:
print("\n" + "=" * 50)
print("TEST 6: Training Step Simulation")
print("=" * 50)

try:
    # Create optimizer
    optimizer = torch.optim.AdamW(flow_model.parameters(), lr=1e-4)
    
    # Training step
    x0 = torch.randn(batch_size, seq_len, latent_dim)
    x1 = torch.randn(batch_size, seq_len, latent_dim)
    genre_ids = torch.tensor([0, 1])
    
    optimizer.zero_grad()
    loss = flow_model(x0, x1, genre_ids)
    loss.backward()
    optimizer.step()
    
    print(f"✓ Training step successful")
    print(f"  - Loss: {loss.item():.6f}")
    print(f"  - Gradients computed: {any(p.grad is not None for p in flow_model.parameters())}")
    
except Exception as e:
    print(f"✗ Training step failed: {e}")
    traceback.print_exc()

## Section 8: Test 7 - End-to-End Pipeline with DAC

Implement complete pipeline: encode audio with DAC, transform latents using flow model, decode back to audio.

In [ ]:
print("\n" + "=" * 50)
print("TEST 7: End-to-End Pipeline")
print("=" * 50)

try:
    # 1. Encode audio with DAC
    audio_input = torch.randn(1, 1, 44100)  # 1 second
    with torch.no_grad():
        z_source, _, _, _, _ = dac_model.encode(audio_input)
    z_source = z_source.transpose(1, 2)  # (B, latent_dim, T) -> (B, T, latent_dim)
    
    print(f"1. Audio encoded: {z_source.shape}")
    
    # 2. Transform with flow model
    target_genre = 1  # Rock
    with torch.no_grad():
        z_transformed = flow_model.sample_euler(z_source, target_genre, num_steps=20)
    
    print(f"2. Latents transformed: {z_transformed.shape}")
    
    # 3. Decode back to audio
    z_transformed = z_transformed.transpose(1, 2)  # Back to DAC format
    with torch.no_grad():
        audio_output = dac_model.decode(z_transformed)
    
    print(f"3. Audio decoded: {audio_output.shape}")
    print(f"\n✓ End-to-end pipeline successful!")
    print(f"  - Input: {audio_input.shape}")
    print(f"  - Output: {audio_output.shape}")
    
except Exception as e:
    print(f"✗ End-to-end pipeline failed: {e}")
    traceback.print_exc()

## Section 9: Test 8 - Memory & Performance Analysis

Measure inference time, time per step, and calculate total model size in MB.

In [ ]:
print("\n" + "=" * 50)
print("TEST 8: Memory & Performance")
print("=" * 50)

try:
    # Measure inference time
    x0 = torch.randn(1, seq_len, latent_dim)
    
    start_time = time.time()
    with torch.no_grad():
        _ = flow_model.sample_euler(x0, 1, num_steps=50)
    inference_time = time.time() - start_time
    
    print(f"✓ Performance test completed")
    print(f"  - Inference time (50 steps): {inference_time:.3f}s")
    print(f"  - Time per step: {inference_time/50*1000:.1f}ms")
    
    # Measure model size
    model_size = sum(p.numel() * p.element_size() for p in flow_model.parameters())
    print(f"  - Model size: {model_size / 1024 / 1024:.2f} MB")
    
except Exception as e:
    print(f"✗ Performance test failed: {e}")
    traceback.print_exc()

## Section 10: Test 9 - Gradient Flow Validation

Check all model parameters for proper gradient computation and identify layers with zero or missing gradients.

In [ ]:
print("\n" + "=" * 50)
print("TEST 9: Gradient Flow Validation")
print("=" * 50)

def test_gradient_flow(model):
    x0 = torch.randn(2, 100, actual_latent_dim, requires_grad=True)
    x1 = torch.randn(2, 100, actual_latent_dim)
    genre_ids = torch.tensor([0, 1])
    
    loss = model(x0, x1, genre_ids)
    loss.backward()
    
    print("\nGradient flow check:")
    grad_count = 0
    zero_grad_count = 0
    
    # Check gradients
    for name, param in model.named_parameters():
        if param.grad is None:
            print(f"  ✗ No gradient for {name}")
        elif param.grad.abs().sum() == 0:
            print(f"  ⚠️  Zero gradient for {name}")
            zero_grad_count += 1
        else:
            grad_count += 1
    
    print(f"\n✓ Gradient flow summary:")
    print(f"  - Layers with good gradients: {grad_count}")
    print(f"  - Layers with zero gradients: {zero_grad_count}")

try:
    test_gradient_flow(flow_model)
except Exception as e:
    print(f"✗ Gradient flow test failed: {e}")
    traceback.print_exc()

## Test Summary & Next Steps

All tests completed! Review results above to verify model functionality.

In [ ]:
print("\n" + "=" * 50)
print("TEST SUMMARY")
print("=" * 50)
print("✓ All tests completed! Check results above.\n")
print("Next steps:")
print("1. Update input_dim in DiT to match DAC latent_dim")
print("2. Prepare your dataset (Slakh2100, FMA, etc.)")
print("3. Create training script with proper data loading")
print("4. Monitor training loss and sample outputs")